# 2026 DL Lab3: Semi-Supervised Intel Image Classification

Before we start, please put **your name** and **SID** in following format: <br>
Hi I'm 陸仁賈, 315831000.

**Your Answer:**    
Hi I'm XXX, XXXXXXXXXX.

## Semi-supervised Intel Image Classification

Train a CNN from scratch using labeled images, then use pseudo-labels to learn from unlabeled images.

**Use PyTorch only. You must implement the model yourself; pre-constructed models, imported architecture blocks, pre-trained weights, extra images, and recovering hidden labels from the public source are not allowed.**

Complete every `TODO` section between the existing comment borders. Keep the data split and class order unchanged. You may adjust the architecture, augmentation and training hyperparameters.

##  Versions of used packages

We will check PyTorch version to make sure everything work properly.

In [ ]:
import sys
import torch
import torchvision
print('python', sys.version.split('\n')[0])
print('torch', torch.__version__)
print('torchvision', torchvision.__version__)

## Random Seed

Set a random seed before creating the model and starting training to make experiments more reproducible. Use the same seed when comparing settings, and try multiple seeds to check stability. Results may still differ across hardware and software versions.


In [ ]:
import random
import numpy as np

SEED = 2026

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True

# Prepare Data

We use [Intel Image Classification](https://www.kaggle.com/datasets/puneet6060/intel-image-classification), provided on Kaggle by Puneet Bansal.

The six classes, in the fixed order used by this notebook, are:
- buildings
- forest
- glacier
- mountain
- sea
- street

Use only the course-provided subset. The original training and test sets are merged and deduplicated before creating four disjoint assignment splits. The additional `seg_pred` pool is not used.

**IMPORTANT: You CANNOT use extra images or look up withheld labels.**

## Unzip Data

Place `Lab3_data_intel_2026.zip` next to this notebook and unzip it. There are 3 folders inside `Lab3_data_intel_2026/`.

- `train/`: 7 subfolders total.
   - `buildings/`, `forest/`, `glacier/`, `mountain/`, `sea/`, `street/`: labeled training images.
   - `unlabel/`: unlabeled training images.
- `val/`: contains 6 folders for the same 6 classes. Labeled validation images for each class.
- `test/`: unclassified images of the testing set.

There are **1800 images in labeled training set.**  

There are **3600 images in unlabeled training set.**

There are **1200 images in validation set.**

There are **3000 images in test set.**  


## Loading the dataset

In [ ]:
data_folder = './Lab3_data_intel_2026'

### Custom dataset

Build a classs inherit `torch.utils.data.Dataset`.  
Implement `__init__`, `__getitem__` and `__len__` 3 functions.  

Some operations could be there: setting location of dataset, the method of reading data, label of dataset or transform of dataset.

See [torch.utils.data.Dataset](https://pytorch.org/docs/stable/data.html#torch.utils.data.Dataset) for more details

In [ ]:
import csv
import os.path as osp
from pathlib import Path
from PIL import Image
import torch
from torch.utils.data import DataLoader, Dataset, TensorDataset
import pandas as pd

CLASS_NAMES = ['buildings', 'forest', 'glacier', 'mountain', 'sea', 'street']
CLASS_TO_IDX = {c:i for i, c in enumerate(CLASS_NAMES)}
VALID_EXTS = {'.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff', '.webp'}

class IntelData(Dataset):
    def __init__(self, root, split='train', mode='train', transform=None, use_unlabel=False):
        self.root = Path(root)
        self.split = split
        self.mode = mode
        self.transform = transform
        self.use_unlabel = use_unlabel

        self.paths = []
        self.labels = []
        self.rel_paths = []

        # Load data from unified CSV files
        if split == 'train' and use_unlabel:
            csv_file = self.root / 'unlabeled_train.csv'
        elif split == 'train':
            csv_file = self.root / 'train.csv'
        elif split == 'val':
            csv_file = self.root / 'val.csv'
        else:  # test
            csv_file = self.root / 'test.csv'

        # Read CSV file using pandas for better handling
        df = pd.read_csv(csv_file)

        for _, row in df.iterrows():
            file_path = self.root / row['file_name']
            self.paths.append(file_path)
            self.rel_paths.append(row['file_name'])

            # Handle labels
            if split == 'test' or (split == 'train' and use_unlabel):
                # No labels for test or unlabeled data
                pass
            else:
                # For labeled data
                if pd.isna(row['groundtruth']) or row['groundtruth'] == '':
                    self.labels.append(-1)  # Invalid label for debugging
                else:
                    self.labels.append(CLASS_TO_IDX[row['groundtruth']])

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, index):
        img_path = self.paths[index]
        img = Image.open(img_path).convert('RGB')
        if self.transform is not None:
            img = self.transform(img)

        if self.mode == 'test' or (self.split == 'train' and self.use_unlabel):
            return img
        label = int(self.labels[index])
        return img, torch.tensor(label, dtype=torch.long)

### Data augmentation

Data augmentation are techniques used to increase the amount of data by adding slightly modified copies of already existing data or newly created synthetic data from existing data.

PyTorch use `torchvision.transforms` to do data augmentation.
[You can see all function here.](https://docs.pytorch.org/vision/main/transforms.html)  

There are some operations may not be necessary for predict, so we should write one for train and one for others.  
**NOTICE**: Use `torchvision.transforms.v2` in this assignment.

In [ ]:
from torchvision.transforms import v2 as transforms
# For TRAIN
########################################################################
# TODO: use transforms.xxx method to do some data augmentation         #
# This one is for training, find the composition to get better result  #
########################################################################
transforms_train = transforms.Compose([
    ...,
    transforms.ToImage(),
    transforms.ToDtype(torch.float32, scale=True),
])
########################################################################
#                           End of your code                           #
########################################################################

# For VAL, TEST
########################################################################
# TODO: use transforms.xxx method to do some data augmentation         #
# This one is for validate and test,                                   #
# NOTICE some operation we usually not use in this part                #
########################################################################
transforms_test = transforms.Compose([
    ...,
    transforms.ToImage(),
    transforms.ToDtype(torch.float32, scale=True),
])
########################################################################
#                           End of your code                           #
########################################################################

### Instantiate dataset

Let's instantiate three `IntelData` class.
+ train_set: for labeled_training.
+ unlabeled_set: for unlabeled_training.
+ dataset_val: for validation.

In [ ]:
train_set       = IntelData(data_folder, split='train', mode='train', transform=transforms_train, use_unlabel=False)
unlabeled_set   = IntelData(data_folder, split='train', mode='test',  transform=transforms_test,  use_unlabel=True)  # train/unlabel
valid_set       = IntelData(data_folder, split='val',   mode='train', transform=transforms_test,  use_unlabel=False)

num_classes = len(CLASS_NAMES)
print("The first image's shape in dataset_train :", train_set[0][0].size())
print("There are", len(train_set), "images in labeled_dataset_train.")
print("There are", len(unlabeled_set), "images in unlabeled_dataset_train.")
print("There are", len(valid_set), "images in dataset_val.")

# Verify the new format by checking a few samples
print("\nVerifying data loading with new CSV format:")
print("Train set - first sample label:", train_set[0][1].item() if len(train_set) > 0 else "No data")
print("Train set - file path:", train_set.rel_paths[0] if len(train_set) > 0 else "No data")
print("Unlabeled set - file path:", unlabeled_set.rel_paths[0] if len(unlabeled_set) > 0 else "No data")
print("Val set - first sample label:", valid_set[0][1].item() if len(valid_set) > 0 else "No data")
print("Val set - file path:", valid_set.rel_paths[0] if len(valid_set) > 0 else "No data")

### DataLoader

`torch.utils.data.DataLoader` define how to sample from `dataset` and some other function like:
+ shuffle : set to `True` to have the data reshuffled at every epoch
+ batch_size : how many samples per batch to load

See [torch.utils.data.DataLoader](https://pytorch.org/docs/stable/data.html#torch.utils.data.DataLoader) for more details

In [ ]:
#####################################################
#            You can adjust batch_size              #
#####################################################
batch_size = 64
num_workers = 0  # Increase to 2 if supported by your runtime
loader_kwargs = dict(batch_size=batch_size, num_workers=num_workers, pin_memory=torch.cuda.is_available())
if num_workers > 0:
    loader_kwargs["persistent_workers"] = True

train_loader = DataLoader(train_set, shuffle=True,  **loader_kwargs)
val_loader   = DataLoader(valid_set, shuffle=False, **loader_kwargs)

Finally! We have made all data prepared.  
Let's go develop our model.

# Self-training

## Step 1: Supervised training

### Implement CNN using PyTorch

Try to use labeled data design and train a deep convolutional network from scratch to predict the class label of a scene image.

**Again, the goal of this assignment is for you to test different convolutional structures. You cannot directly use the blocks/architectures of pre-trained models.**

In [ ]:
from torch.nn.modules.conv import Conv2d
import torch.nn as nn
import torch.nn.functional as F
class YourCNNModel(nn.Module):
    def __init__(self, num_classes=6):
        super().__init__()
        ########################################################################
        # TODO: use nn.xxx method to generate a CNN model part                 #
        ########################################################################
        self.conv = ... # You can change the variable name
        ########################################################################
        #                           End of your code                           #
        ########################################################################

    def forward(self, x):
        assert isinstance(x, torch.Tensor), "Input should be a torch Tensor"
        assert x.dim() == 4, "Input should be NCHW format"
        ########################################################################
        # TODO: forward your model and get output                              #
        ########################################################################
        out = ...
        ########################################################################
        #                           End of your code                           #
        ########################################################################
        return out

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

In [ ]:
model = YourCNNModel(num_classes=num_classes)
model = model.to(device)

We have made our model!  
Next, PyTorch also provide many utility function(loss, optmizer...etc).  
You can define them in one-line.

### Define loss and optimizer

[Optimizers in pytorch](https://docs.pytorch.org/docs/stable/optim.html)  
[CrossEntropyLoss in pytorch](https://docs.pytorch.org/docs/stable/generated/torch.nn.CrossEntropyLoss.html)

In [ ]:
import torch.nn as nn
import torch.optim as optim
################################################################################
# TODO: Define loss and optmizer functions                                     #
# Try any loss or optimizer function and learning rate to get better result    #
# hint: torch.nn and torch.optim                                               #
################################################################################
criterion = ...
optimizer = ...
################################################################################
#                               End of your code                               #
################################################################################
criterion = criterion.to(device)

### Train the model

#### Train function
Let's define train function.  
It will iterate input data 1 epoch and update model with optmizer.  
Finally, calculate mean loss and total accuracy.

Hint: [torch.max()](https://pytorch.org/docs/stable/generated/torch.max.html#torch-max) or [torch.argmax()](https://pytorch.org/docs/stable/generated/torch.argmax.html)

In [ ]:
from tqdm import tqdm
import torch

def train(input_data, model, criterion, optimizer, epoch=None, total_epochs=None):
    '''
    Argement:
    input_data -- iterable data, typr torch.utils.data.Dataloader is prefer
    model -- nn.Module, model contain forward to predict output
    criterion -- loss function, used to evaluate goodness of model
    optimizer -- optmizer function, method for weight updating
    '''
    model.train()
    total_loss = 0.0
    total_count = 0
    acc_count = 0

    desc = f"Train | epoch {epoch}/{total_epochs}" if epoch is not None else "Train"
    pbar = tqdm(input_data, desc=desc, leave=False)

    for images, labels in pbar:
        images = images.to(device)
        labels = labels.to(device)

        ########################################################################
        # TODO: Forward, backward and optimize                                 #
        # 1. zero the parameter gradients                                      #
        # 2. process input through the network                                 #
        # 3. compute the loss                                                  #
        # 4. propagate gradients back into the network's parameters            #
        # 5. Update the weights of the network                                 #
        ########################################################################
        pass
        ########################################################################
        #                           End of your code                           #
        ########################################################################


        ########################################################################
        # TODO: Get the counts of correctly classified images                  #
        # 1. get the model predicted result                                    #
        # 2. sum the number of this batch predicted images                     #
        # 3. sum the number of correctly classified                            #
        # 4. accumulate loss weighted by batch size                            #
        # dimension of outputs: [batch_size, number of classes]                #
        # Hint 1: use outputs.detach() if a detached tensor is needed          #
        # Hint 2: use torch.max()                                              #
        ########################################################################
        predicted = ...
        total_count += ...
        acc_count += ...
        total_loss += ...
        ########################################################################
        #                           End of your code                           #
        ########################################################################

        running_acc = acc_count / total_count if total_count else 0.0
        lr = optimizer.param_groups[0]['lr']
        pbar.set_postfix(loss=f"{loss.item():.4f}", acc=f"{running_acc:.4f}", lr=f"{lr:.6f}")

    acc  = acc_count / total_count if total_count else 0.0
    loss = total_loss / total_count if total_count else 0.0
    return acc, loss

#### Validate function
Next part is validate function.  
It works as training function without optmizer and weight-updating part.

In [ ]:
def val(input_data, model, criterion, epoch=None, total_epochs=None):
    model.eval()
    total_loss = 0.0
    total_count = 0
    acc_count = 0

    desc = f"Val   | epoch {epoch}/{total_epochs}" if epoch is not None else "Val"
    pbar = tqdm(input_data, desc=desc, leave=False)

    with torch.no_grad():
        for images, labels in pbar:
            images = images.to(device)
            labels = labels.to(device)

            ####################################################################
            # TODO: Get the predicted result and loss                          #
            # 1. process input through the network                             #
            # 2. compute the loss                                              #
            # 3. get the model predicted result                                #
            # 4. get the counts of correctly classified images                 #
            # 5. accumulate loss weighted by batch size                         #
            ####################################################################
            pass
            
            total_count += ...
            acc_count += ...
            total_loss += ...
            ####################################################################
            #                         End of your code                         #
            ####################################################################

            running_acc = acc_count / total_count if total_count else 0.0
            pbar.set_postfix(loss=f"{loss.item():.4f}", acc=f"{running_acc:.4f}")

    acc  = acc_count / total_count if total_count else 0.0
    loss = total_loss / total_count if total_count else 0.0
    return acc, loss

#### Training in a loop
Call train and test function in a loop.  
Take a break and wait.

In [ ]:
################################################################################
#     You can adjust those hyper parameters to loop for max_epochs times       #
################################################################################
max_epochs = 30
log_interval = 10

SUPERVISED_CKPT = 'supervised.pt'
best_acc = -1.0

train_acc_list = []
train_loss_list = []
val_acc_list = []
val_loss_list = []

for epoch in range(1, max_epochs + 1):
    train_acc, train_loss = train(train_loader, model, criterion, optimizer, epoch=epoch, total_epochs=max_epochs)
    val_acc, val_loss     = val(val_loader, model, criterion, epoch=epoch, total_epochs=max_epochs)

    if val_acc > best_acc:
        best_acc = val_acc
        torch.save(model.state_dict(), SUPERVISED_CKPT)

    train_acc_list.append(train_acc)
    train_loss_list.append(train_loss)
    val_acc_list.append(val_acc)
    val_loss_list.append(val_loss)

    if epoch % log_interval == 0:
        lr = optimizer.param_groups[0]['lr']
        print('=' * 20, f'Epoch {epoch}/{max_epochs}', '=' * 20)
        print('Train Acc: {:.6f} | Train Loss: {:.6f}'.format(train_acc, train_loss))
        print('  Val Acc: {:.6f} |   Val Loss: {:.6f} | LR: {:.6f}'.format(val_acc, val_loss, lr))

################################################################################
#                               End of your code                               #
################################################################################

#### Visualize accuracy and loss

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(12, 4))
plt.plot(range(len(train_loss_list)), train_loss_list)
plt.plot(range(len(val_loss_list)), val_loss_list)
plt.legend(['train', 'val'])
plt.title('Loss')
plt.show()

plt.figure(figsize=(12, 4))
plt.plot(range(len(train_acc_list)), train_acc_list)
plt.plot(range(len(val_acc_list)), val_acc_list)
plt.legend(['train', 'val'])
plt.title('Acc')
plt.show()

finish training your classifier, next you should use this classifer to predict unlabel images with pseduo label.

## Step2: Use unlabeled data to enhance model performance

In [ ]:
# load the trained classifier weights
ckpt = torch.load(SUPERVISED_CKPT, map_location=device, weights_only=True)
model.load_state_dict(ckpt)

In [ ]:
# Keep image indices instead of loading all unlabeled images into memory.
# Rerun this cell when restarting Step 2.
unlabeled_indices = list(range(len(unlabeled_set)))

class PseudoLabelData(Dataset):
    def __init__(self, paths, labels, transform):
        self.paths = paths
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, index):
        image_path = self.paths[index]
        with Image.open(image_path) as image:
            image = image.convert('RGB')
            image = self.transform(image)

        label = self.labels[index]
        label = torch.tensor(label, dtype=torch.long)
        return image, label

print('Unlabeled pool size:', len(unlabeled_indices))

### Use the trained classifier to generate pseudo-labels.

In [ ]:
from torch.utils.data import ConcatDataset, Subset

###########################################################
#   You can adjust the threshold to get better results!   #
###########################################################
def get_pseudo_labels(model, threshold=0.9):
    model.eval()
    paths_keep = []
    labels_keep = []
    remove_indices = []

    remaining_set = Subset(unlabeled_set, unlabeled_indices)

    loader = DataLoader(
        remaining_set,
        shuffle=False,
        **loader_kwargs
    )
    offset = 0

    ################################################################################
    # TODO: Generate pseudo-labels for the remaining unlabeled images.             #
    # 1. Forward the images through the model using torch.no_grad().               #
    # 2. Apply softmax to obtain class probabilities.                              #
    # 3. Keep predictions whose confidence reaches the threshold.                  #
    # 4. Collect the accepted image paths and predicted labels.                    #
    # 5. Add accepted pool positions to remove_indices for later removal.          #
    ################################################################################
    with torch.no_grad():
        for images in tqdm(loader, desc='Pseudo-labeling', leave=False):
            images = images.to(device)

            ...
            
            offset += len(images)
    ################################################################################
    #                               End of your code                               #
    ################################################################################

    # Delete backwards so earlier positions do not shift.
    for index in reversed(remove_indices):
        del unlabeled_indices[index]

    pseudo_dataset = PseudoLabelData(
        paths_keep,
        labels_keep,
        transforms_train
    )

    taken = len(pseudo_dataset)
    remaining = len(unlabeled_indices)

    print(f'Labeled {taken} images this round. Remaining unlabeled: {remaining}')

    return pseudo_dataset, taken

### Redefine your optimizer if you want

In [ ]:
################################################################################
# TODO: Define loss and optmizer functions                                     #
# Try any loss or optimizer function and learning rate to get better result    #
# hint: torch.nn and torch.optim                                               #
################################################################################
criterion = ...
optimizer = ...
################################################################################
#                               End of your code                               #
################################################################################
criterion = criterion.to(device)

### Train the model

Every `N` epochs, use `get_pseudo_labels` to label the remaining unlabeled images.

Keep high-confidence predictions and remove those images from the unlabeled pool.

Combine the original labeled data with **all accumulated pseudo-labeled data**, then build a new DataLoader.

Previously accepted pseudo-labels stay unchanged.


In [ ]:
################################################################################
# You can adjust the number of epochs, update interval, and threshold.         #
################################################################################
n_epochs = 50
N = 10  # Add new pseudo-labels every N epochs.
log_interval = 10 # Log every N epochs like supervised training.
threshold = 0.9
SELF_TRAIN_CKPT = 'self_training.pt'

best_acc, _ = val(val_loader, model, criterion)
torch.save(model.state_dict(), SELF_TRAIN_CKPT)

current_train_dataset = train_set
train_loader_ssl = train_loader
all_pseudo_datasets = []
total_pseudo = 0

for epoch in range(n_epochs):
    ################################################################################
    # TODO: Construct the training dataset for semi-supervised learning.           #
    # 1. Every N epochs, call get_pseudo_labels on the remaining unlabeled data.   #
    # 2. Add the newly accepted pseudo-label dataset to all_pseudo_datasets.       #
    # 3. Combine train_set with ALL accumulated pseudo-label datasets.             #
    # 4. Create a new DataLoader for the combined dataset.                         #
    # Hint: ConcatDataset and DataLoader.                                          #
    ################################################################################
    if epoch % N == 0:
        ...
    ################################################################################
    #                               End of your code                               #
    ################################################################################

    train_acc, train_loss = train(
        train_loader_ssl,
        model,
        criterion,
        optimizer,
        epoch=epoch + 1,
        total_epochs=n_epochs
    )
    
    valid_acc, valid_loss = val(
        val_loader,
        model,
        criterion,
        epoch=epoch + 1,
        total_epochs=n_epochs
    )
    
    lr = optimizer.param_groups[0]['lr']

    if (epoch + 1) % log_interval == 0:
        print('=' * 20, f'Epoch {epoch+1}/{n_epochs}', '=' * 20)
        print('Train Acc: {:.6f} | Train Loss: {:.6f}'.format(train_acc, train_loss))
        print('  Val Acc: {:.6f} |   Val Loss: {:.6f} | LR: {:.6f}'.format(valid_acc, valid_loss, lr))
        print('Dataset Size: {} (original: {}, pseudo: {})'.format(len(current_train_dataset), len(train_set), total_pseudo))

    if valid_acc > best_acc:
        best_acc = valid_acc
        torch.save(model.state_dict(), SELF_TRAIN_CKPT)
        if (epoch + 1) % log_interval != 0: 
            print('[{:03d}/{:03d}] saving model with acc {:.3f}'.format(epoch + 1, n_epochs, best_acc))

print('Best validation accuracy:', best_acc)

### Predict Result

**How to upload**

1. Click the folder icon in the left hand side of Colab.
2. Right click "result.csv". Select "Download".
3. To kaggle. Click "Submit Predictions"
4. Upload the result.csv.

In [ ]:
# if you wanna load previous best model
ckpt = torch.load('self_training.pt', map_location=device)
model.load_state_dict(ckpt)

In [ ]:
test_set = IntelData(data_folder, split='test', mode='test', transform=transforms_test)
test_loader = DataLoader(
    test_set,
    batch_size=batch_size,
    num_workers=num_workers,
    shuffle=False,
    pin_memory=torch.cuda.is_available()
)

In [ ]:
def predict(input_data, model):
    model.eval()
    output_list = []
    with torch.no_grad():
        for images in input_data:
            images = images.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs, 1)
            output_list.extend(predicted.to('cpu').numpy().tolist())
    return output_list

In [ ]:
pred_indices = predict(test_loader, model)

with open('result.csv', 'w', newline='') as csvFile:
    writer = csv.DictWriter(csvFile, fieldnames=['ID', 'label'])
    writer.writeheader()
    for filename, pred in zip(test_set.paths, pred_indices):
        filename = osp.basename(filename)  # Extract just the filename
        writer.writerow({'ID': filename, 'label': CLASS_NAMES[int(pred)]})

print("Saved predictions to result.csv")